In [1]:
!pip install -q -U transformers accelerate peft bitsandbytes datasets trl gradio requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 11.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
import os, json, re, random
import requests
import torch
from datasets import Dataset

torch.manual_seed(42)
random.seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)
assert DEVICE == "cuda", "Enable a GPU runtime: Runtime > Change runtime type > T4 GPU"


Using device: cuda


In [3]:
DRUGS = [
    "ibuprofen", "acetaminophen", "amoxicillin", "metformin", "atorvastatin",
    "omeprazole", "lisinopril", "losartan", "sertraline", "albuterol",
    "azithromycin", "amlodipine", "metoprolol", "gabapentin", "hydrochlorothiazide",
    "levothyroxine", "prednisone", "furosemide", "warfarin", "insulin glargine",
]

FIELDS = {
    "indications_and_usage": "What is {drug} used for?",
    "dosage_and_administration": "What is the recommended dosage for {drug}?",
    "warnings": "What are the warnings associated with {drug}?",
    "adverse_reactions": "What are the side effects of {drug}?",
    "contraindications": "When should {drug} not be used?",
}

def clean_text(text):
    text = re.sub(r"\s+", " ", text).strip()
    # Trim overly long label sections so training stays fast
    return text[:1200]

def fetch_label(drug_name):
    url = "https://api.fda.gov/drug/label.json"
    params = {"search": f'openfda.generic_name:"{drug_name}"', "limit": 1}
    r = requests.get(url, params=params, timeout=20)
    if r.status_code != 200:
        return None
    results = r.json().get("results")
    return results[0] if results else None

raw_records = []
for drug in DRUGS:
    try:
        label = fetch_label(drug)
        if not label:
            print(f"  no label found for {drug}, skipping")
            continue
        for field, question_template in FIELDS.items():
            values = label.get(field)
            if values:
                answer = clean_text(values[0])
                if len(answer) > 40:
                    raw_records.append({
                        "drug": drug,
                        "instruction": question_template.format(drug=drug.title()),
                        "output": answer,
                    })
        print(f"  collected sections for {drug}")
    except Exception as e:
        print(f"  error fetching {drug}: {e}")

print(f"\nTotal instruction/response pairs collected: {len(raw_records)}")

  collected sections for ibuprofen
  collected sections for acetaminophen
  collected sections for amoxicillin
  collected sections for metformin
  collected sections for atorvastatin
  collected sections for omeprazole
  collected sections for lisinopril
  collected sections for losartan
  collected sections for sertraline
  collected sections for albuterol
  collected sections for azithromycin
  collected sections for amlodipine
  collected sections for metoprolol
  collected sections for gabapentin
  collected sections for hydrochlorothiazide
  collected sections for levothyroxine
  collected sections for prednisone
  collected sections for furosemide
  collected sections for warfarin
  collected sections for insulin glargine

Total instruction/response pairs collected: 82


In [4]:
os.makedirs("data", exist_ok=True)
with open("data/pharma_raw.jsonl", "w") as f:
    for r in raw_records:
        f.write(json.dumps(r) + "\n")

print(raw_records[0])

{'drug': 'ibuprofen', 'instruction': 'What is Ibuprofen used for?', 'output': 'Uses temporarily relieves minor aches and pains due to: headache toothache backache menstrual cramps the common cold muscular aches minor pain of arthritis temporarily reduces fever'}


In [5]:
PROMPT_TEMPLATE = (
    "### Instruction:\n{instruction}\n\n### Response:\n{output}"
)

def format_example(example):
    example["text"] = PROMPT_TEMPLATE.format(
        instruction=example["instruction"], output=example["output"]
    )
    return example

random.shuffle(raw_records)
split_idx = int(len(raw_records) * 0.9)
train_records = raw_records[:split_idx]
val_records = raw_records[split_idx:]

train_ds = Dataset.from_list(train_records).map(format_example)
val_ds = Dataset.from_list(val_records).map(format_example)

print(f"Train size: {len(train_ds)}, Val size: {len(val_ds)}")
print(train_ds[0]["text"])


Map:   0%|          | 0/73 [00:00<?, ? examples/s]

Map:   0%|          | 0/9 [00:00<?, ? examples/s]

Train size: 73, Val size: 9
### Instruction:
What are the warnings associated with Hydrochlorothiazide?

### Response:
WARNINGS General Lisinopril Anaphylactoid and Possibly Related Reactions: Presumably because angiotensin-converting enzyme inhibitors affect the metabolism of eicosanoids and polypeptides, including endogenous bradykinin, patients receiving ACE inhibitors (including lisinopril and hydrochlorothiazide tablets) may be subject to a variety of adverse reactions, some of them serious. Head and Neck Angioedema: Angioedema of the face, extremities, lips, tongue, glottis and/or larynx has been reported rarely in patients treated with angiotensin converting enzyme inhibitors, including lisinopril. This may occur at any time during treatment. ACE inhibitors have been associated with a higher rate of angioedema in Black than in non-Black patients. In such cases lisinopril and hydrochlorothiazide tablets should be promptly discontinued and appropriate therapy and monitoring should

In [6]:
!pip install -q -U bitsandbytes

In [7]:
!pip install -U bitsandbytes accelerate

In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
base_model.config.use_cache = False

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [9]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

base_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    r=16,                 # rank — higher = more capacity, more params
    lora_alpha=32,        # scaling factor
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # attention layers
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


In [10]:
MAX_LEN = 512

def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_LEN,
        padding="max_length",
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

train_tok = train_ds.map(tokenize, remove_columns=train_ds.column_names)
val_tok = val_ds.map(tokenize, remove_columns=val_ds.column_names)


Map:   0%|          | 0/73 [00:00<?, ? examples/s]

Map:   0%|          | 0/9 [00:00<?, ? examples/s]

In [11]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./pharma-lora-checkpoints",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,1.451916,1.286001
2,1.335846,1.197915
3,1.262770,1.163166


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=15, training_loss=1.350177033742269, metrics={'train_runtime': 76.3488, 'train_samples_per_second': 2.868, 'train_steps_per_second': 0.196, 'total_flos': 699018051059712.0, 'train_loss': 1.350177033742269, 'epoch': 3.0})

In [12]:
ADAPTER_DIR = "./pharma-lora-adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Adapter saved to", ADAPTER_DIR)

# Optional: merge adapters into the base model for standalone deployment
merged_model = model.merge_and_unload()
merged_model.save_pretrained("./pharma-merged-model")
tokenizer.save_pretrained("./pharma-merged-model")
print("Merged model saved to ./pharma-merged-model")

Adapter saved to ./pharma-lora-adapter


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:377: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved to ./pharma-merged-model


In [13]:
def generate(prompt_model, instruction, max_new_tokens=150):
    prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = prompt_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    return text.split("### Response:")[-1].strip()

test_questions = [q["instruction"] for q in val_records[:3]] or [
    "What is Metformin used for?",
    "What are the side effects of Ibuprofen?",
]

for q in test_questions:
    print("Q:", q)
    print("Fine-tuned model:", generate(model, q))
    print("-" * 80)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is Furosemide used for?


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Fine-tuned model: Furosemide is a medication used to treat various conditions such as heart failure, hypertension, and edema. It is also used to treat certain types of kidney stones and to prevent the buildup of salt and water in the body.
--------------------------------------------------------------------------------
Q: When should Metformin not be used?


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Fine-tuned model: Metformin should not be used in patients with type 2 diabetes who have a history of pancreatitis or have a history of pancreatitis in the past 12 months. It is also not recommended in patients with type 2 diabetes who have a history of gastrointestinal bleeding or have a history of gastrointestinal bleeding in the past 12 months.
--------------------------------------------------------------------------------
Q: When should Atorvastatin not be used?
Fine-tuned model: Atorvastatin should not be used in patients with severe liver disease, as it can cause liver damage. It is also not recommended for patients with uncontrolled high cholesterol levels, as it may not be effective in lowering them.
--------------------------------------------------------------------------------


In [ ]:
import gradio as gr
DISCLAIMER = (
    "⚠️ Educational demo only — not medical advice. "
    "Always consult official FDA labeling or a licensed professional."
)

def chat_fn(message, history):
    response = generate(model, message)
    return response

demo = gr.ChatInterface(
    fn=chat_fn,
    title="Pharma Drug Information Assistant (LoRA fine-tuned TinyLlama)",
    description=DISCLAIMER,
    examples=[
        "What is Amoxicillin used for?",
        "What is the recommended dosage for Atorvastatin?",
        "What are the warnings for Warfarin?",
    ],
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ddc09eb637e93560d9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
